# GammaNet: effect of removing top-down layers

Deze notebook vergelijkt hetzelfde checkpoint in twee condities:

1. **full_td**: normale `VGG16GammaNetV2` forward met bottom-up + top-down pass.
2. **no_td**: tijdelijke monkeypatch waarbij alleen de bottom-up `h*_exc` recurrente lagen draaien en de top-down pass wordt overgeslagen.

De analyse meet vooral in `h1_exc`, omdat je wilt weten of top-down feedback de contourrepresentatie in die onderliggende laag verbetert.

Belangrijk: deze notebook verandert je modelbestand/checkpoint niet permanent. De no-topdown forward wordt alleen tijdelijk in Python gebruikt.

In [ ]:
# ============================================================
# 1. Imports, paths, settings
# ============================================================

import os, re, sys, types
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ------------------------------------------------------------
# Paths - pas eventueel aan
# ------------------------------------------------------------
PROJECT_ROOT = Path("/home/yentl/pytorch_gammanet")
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoint_epoch_40.pt"
IMAGE_DIR = PROJECT_ROOT / "Images_all_layers"
MASK_DIR = PROJECT_ROOT / "contour_masks"

OUTPUT_DIR = PROJECT_ROOT / "outputs_no_topdown_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
INPUT_SIZE = (320, 320)
N_IMAGES_PER_CONTOUR = 25   # zet tijdelijk op 3 om snel te testen
JITTER_TO_USE = 0
CONTRAST_TO_USE = "high"
MODEL_TIMESTEPS = 4

USE_ABS_ACTIVATION = True
TOP_ACTIVATION_PERCENTILE = 90

BOTTOM_UP_LAYERS = ["h0_exc", "h1_exc", "h2_exc", "h3_exc", "h4_exc"]
ANALYSIS_LAYER = "h1_exc"
MODEL_CONDITIONS = ["full_td", "no_td"]

for sub in [
    "csv",
    "plots",
    "plots/mean_layer_maps",
    "plots/mask_metrics",
    "plots/td_effect",
    "plots/temporal_dynamics",
    "plots/mask_checks",
]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Output:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Load model
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from gammanet.models.vgg16_gammanet_v2 import VGG16GammaNetV2

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

if "config" in checkpoint and "model" in checkpoint["config"]:
    model_config = checkpoint["config"]["model"]
elif "model_config" in checkpoint:
    model_config = checkpoint["model_config"]
else:
    raise KeyError("Could not find model config in checkpoint.")

model = VGG16GammaNetV2(model_config)

state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", None))
if state_dict is None:
    raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model.to(DEVICE)
model.eval()
model.timesteps = MODEL_TIMESTEPS

print("Model timesteps:", model.timesteps)
print("skip_connections:", getattr(model, "skip_connections", None))


In [ ]:
# ============================================================
# 3. Dataset parsing and J000 image selection
# ============================================================

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
])


def normalize_contour_label(label):
    lower = label.strip().lower()
    if lower in ["c", "ccontour", "c_contour"]:
        return "C"
    if lower in ["straight", "line", "straightline", "straight_line"]:
        return "straight"
    return label


def parse_image_filename(path):
    parts = path.stem.split("_")
    if len(parts) < 6:
        return None
    contour_type = normalize_contour_label(parts[0])
    if contour_type not in ["C", "straight"]:
        return None
    return {
        "filename": path.name,
        "path": str(path),
        "contour_type": contour_type,
        "contrast": parts[1],
        "quadrant": parts[2],
        "position": int(parts[3]),
        "jitter": int(parts[4].replace("J", "")),
        "stimulus_id": int(parts[5]),
    }


def load_image_tensor(path, requires_grad=False):
    pil_img_original = Image.open(path).convert("RGB")
    pil_img_model = pil_img_original.resize(INPUT_SIZE)
    tensor = transform(pil_img_original).unsqueeze(0).to(DEVICE)
    tensor.requires_grad_(requires_grad)
    return pil_img_model, tensor

rows = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    try:
        parsed = parse_image_filename(path)
        if parsed is not None:
            rows.append(parsed)
    except Exception as e:
        print("Skipped", path.name, e)

dataset_df = pd.DataFrame(rows)
print("Available images:")
display(dataset_df.groupby(["contour_type", "contrast", "jitter"]).size().reset_index(name="n").head(20))

selected_df = (
    dataset_df[
        (dataset_df["jitter"] == JITTER_TO_USE) &
        (dataset_df["contrast"] == CONTRAST_TO_USE) &
        (dataset_df["contour_type"].isin(["C", "straight"]))
    ]
    .sort_values(["contour_type", "quadrant", "position", "stimulus_id"])
    .groupby("contour_type", group_keys=False)
    .head(N_IMAGES_PER_CONTOUR)
    .reset_index(drop=True)
)

print("Selected images:")
print(selected_df.groupby("contour_type").size())
display(selected_df[["filename", "contour_type", "contrast", "quadrant", "position", "jitter"]].head())

if selected_df.groupby("contour_type").size().min() < N_IMAGES_PER_CONTOUR:
    print("WAARSCHUWING: minder afbeeldingen gevonden dan N_IMAGES_PER_CONTOUR voor minstens één contour.")


In [ ]:
# ============================================================
# 4. Basic helpers
# ============================================================

def reset_hidden_and_forward(model, img_tensor):
    model.reset_hidden_states()
    with torch.no_grad():
        return model(img_tensor)


def get_state(model, layer_name):
    state = getattr(model, layer_name, None)
    if state is None:
        raise ValueError(f"{layer_name} is None. Run model first or check layer name.")
    return state


def normalize_for_plot(x, eps=1e-8):
    x = np.asarray(x)
    x = x - np.nanmin(x)
    return x / (np.nanmax(x) + eps)


def resize_map_to_image(fmap, pil_img):
    fmap_t = torch.tensor(fmap, dtype=torch.float32)[None, None]
    resized = F.interpolate(
        fmap_t,
        size=pil_img.size[::-1],
        mode="bilinear",
        align_corners=False,
    )
    return resized[0, 0].numpy()


def all_channel_map(state, use_abs=True):
    x = state.detach()
    if use_abs:
        x = x.abs()
    return x.mean(dim=1)[0].cpu().numpy()


def single_channel_map(state, channel):
    return state.detach().cpu()[0, int(channel)].numpy()


def load_true_contour_mask(row, target_size):
    mask_path = MASK_DIR / row["filename"]
    if not mask_path.exists():
        raise FileNotFoundError(f"No mask found for {row['filename']} at {mask_path}")
    mask = Image.open(mask_path).convert("L")
    mask = mask.resize(target_size, resample=Image.NEAREST)
    return np.asarray(mask) > 0


In [ ]:
# ============================================================
# 5. No-topdown forward pass
# ============================================================

# Deze functie kopieert de bottom-up pass uit VGG16GammaNetV2.forward,
# maar stopt vóór de top-down pass. De recurrente h*_exc fGRU-lagen blijven dus bestaan.
# Alleen td_fgru_4_to_3, td_fgru_3_to_2, td_fgru_2_to_1, td_fgru_1_to_0 en td_fgru_* worden overgeslagen.


def bottom_up_only_forward(self, x):
    batch_size = x.shape[0]
    device = x.device

    if self.h0_exc is None or self.h0_exc.shape[0] != batch_size:
        self.init_hidden_states(batch_size, x.shape[2], x.shape[3], device)

    self.temporal_activity = {layer: [] for layer in BOTTOM_UP_LAYERS}

    for t in range(self.timesteps):
        # h0
        x1 = self.block1_conv(x)
        if self.use_separate_ei_states:
            self.h0_exc, self.h0_inh, _ = self.fgru_0(x1, self.h0_exc, self.h0_inh)
            aligned_0 = self.align_0(self.h0_exc)
        else:
            self.h0_exc, _, _ = self.fgru_0(x1, self.h0_exc)
            aligned_0 = self.align_0(self.h0_exc)
        self.temporal_activity["h0_exc"].append(self.h0_exc.detach().cpu())
        x1 = self.pool1(aligned_0)

        # h1
        x2 = self.block2_conv(x1)
        if self.use_separate_ei_states:
            self.h1_exc, self.h1_inh, _ = self.fgru_1(x2, self.h1_exc, self.h1_inh)
            aligned_1 = self.align_1(self.h1_exc)
        else:
            self.h1_exc, _, _ = self.fgru_1(x2, self.h1_exc)
            aligned_1 = self.align_1(self.h1_exc)
        self.temporal_activity["h1_exc"].append(self.h1_exc.detach().cpu())
        x2 = self.pool2(aligned_1)

        # h2
        x3 = self.block3_conv(x2)
        if self.use_separate_ei_states:
            self.h2_exc, self.h2_inh, _ = self.fgru_2(x3, self.h2_exc, self.h2_inh)
            aligned_2 = self.align_2(self.h2_exc)
        else:
            self.h2_exc, _, _ = self.fgru_2(x3, self.h2_exc)
            aligned_2 = self.align_2(self.h2_exc)
        self.temporal_activity["h2_exc"].append(self.h2_exc.detach().cpu())
        x3 = self.pool3(aligned_2)

        # h3
        x4 = self.block4_conv(x3)
        if self.use_separate_ei_states:
            self.h3_exc, self.h3_inh, _ = self.fgru_3(x4, self.h3_exc, self.h3_inh)
            aligned_3 = self.align_3(self.h3_exc)
        else:
            self.h3_exc, _, _ = self.fgru_3(x4, self.h3_exc)
            aligned_3 = self.align_3(self.h3_exc)
        self.temporal_activity["h3_exc"].append(self.h3_exc.detach().cpu())
        x4 = self.pool4(aligned_3)

        # h4
        x5 = self.block5_conv(x4)
        if self.use_separate_ei_states:
            self.h4_exc, self.h4_inh, _ = self.fgru_4(x5, self.h4_exc, self.h4_inh)
        else:
            self.h4_exc, _, _ = self.fgru_4(x5, self.h4_exc)
        self.temporal_activity["h4_exc"].append(self.h4_exc.detach().cpu())

    # Voor compatibiliteit maken we een output uit h0_exc.
    # De analyse gebruikt vooral h1_exc, dus de output zelf is niet centraal.
    return self.output_projection(self.h0_exc)


class NoTopDownForward:
    """Tijdelijke context manager: vervangt model.forward door bottom_up_only_forward."""

    def __init__(self, model):
        self.model = model
        self.original_forward = None

    def __enter__(self):
        self.original_forward = self.model.forward
        self.model.forward = types.MethodType(bottom_up_only_forward, self.model)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.model.forward = self.original_forward
        return False


def run_condition_forward(model, img_tensor, condition):
    model.reset_hidden_states()
    with torch.no_grad():
        if condition == "full_td":
            return model(img_tensor)
        elif condition == "no_td":
            with NoTopDownForward(model):
                return model(img_tensor)
        else:
            raise ValueError(f"Unknown condition: {condition}")


In [ ]:
# ============================================================
# 6. Sanity check: compare state shapes full_td vs no_td
# ============================================================

example_row = selected_df.iloc[0]
pil_img, img_tensor = load_image_tensor(example_row["path"])

for condition in MODEL_CONDITIONS:
    run_condition_forward(model, img_tensor, condition)
    print("\n", condition)
    for layer in BOTTOM_UP_LAYERS:
        state = get_state(model, layer)
        print(layer, tuple(state.shape), "mean abs", state.abs().mean().item())


In [ ]:
# ============================================================
# 7. Visual mask check
# ============================================================

for contour_type in ["C", "straight"]:
    row = selected_df[selected_df["contour_type"] == contour_type].iloc[0]
    pil_img, _ = load_image_tensor(row["path"])
    mask = load_true_contour_mask(row, target_size=pil_img.size)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(pil_img)
    axes[0].set_title("Stimulus")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("True contour mask")
    axes[1].axis("off")

    axes[2].imshow(pil_img)
    axes[2].imshow(mask, cmap="Reds", alpha=0.45)
    axes[2].set_title("Mask overlay")
    axes[2].axis("off")

    plt.suptitle(row["filename"])
    plt.tight_layout()
    save_path = OUTPUT_DIR / "plots" / "mask_checks" / f"mask_check_{contour_type}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# 8. Collect mean final states per condition x contour x layer
# ============================================================

# mean_states[condition][contour][layer] = mean final state over selected images
mean_states = {
    condition: {contour: {} for contour in ["C", "straight"]}
    for condition in MODEL_CONDITIONS
}

final_sum = {
    condition: {contour: {layer: None for layer in BOTTOM_UP_LAYERS} for contour in ["C", "straight"]}
    for condition in MODEL_CONDITIONS
}
final_count = {condition: {contour: 0 for contour in ["C", "straight"]} for condition in MODEL_CONDITIONS}

temporal_sum = {
    condition: {contour: {layer: None for layer in BOTTOM_UP_LAYERS} for contour in ["C", "straight"]}
    for condition in MODEL_CONDITIONS
}
temporal_count = {condition: {contour: 0 for contour in ["C", "straight"]} for condition in MODEL_CONDITIONS}

for condition in MODEL_CONDITIONS:
    print("\n========================================")
    print("Condition:", condition)

    for _, row in tqdm(selected_df.iterrows(), total=len(selected_df)):
        contour = row["contour_type"]
        pil_img, img_tensor = load_image_tensor(row["path"])
        run_condition_forward(model, img_tensor, condition)

        for layer in BOTTOM_UP_LAYERS:
            state = get_state(model, layer).detach().cpu()
            if final_sum[condition][contour][layer] is None:
                final_sum[condition][contour][layer] = state.clone()
            else:
                final_sum[condition][contour][layer] += state

            if hasattr(model, "temporal_activity") and layer in model.temporal_activity:
                stack = torch.stack(model.temporal_activity[layer], dim=0)  # [T, B, C, H, W]
                if temporal_sum[condition][contour][layer] is None:
                    temporal_sum[condition][contour][layer] = stack.clone()
                else:
                    temporal_sum[condition][contour][layer] += stack

        final_count[condition][contour] += 1
        temporal_count[condition][contour] += 1

for condition in MODEL_CONDITIONS:
    for contour in ["C", "straight"]:
        for layer in BOTTOM_UP_LAYERS:
            mean_states[condition][contour][layer] = (
                final_sum[condition][contour][layer] / final_count[condition][contour]
            )

print("Done.")


In [ ]:
# ============================================================
# 9. Plot mean all-channel layer maps, old 3-panel style
# ============================================================

MEAN_MAP_DIR = OUTPUT_DIR / "plots" / "mean_layer_maps"
MEAN_MAP_DIR.mkdir(parents=True, exist_ok=True)


def plot_mean_state_overlay(condition, contour_type, layer_name, use_abs=True, alpha=0.55):
    row = selected_df[selected_df["contour_type"] == contour_type].iloc[0]
    pil_img, _ = load_image_tensor(row["path"])

    state = mean_states[condition][contour_type][layer_name]
    fmap = all_channel_map(state, use_abs=use_abs)
    fmap_resized = resize_map_to_image(fmap, pil_img)
    fmap_norm = normalize_for_plot(fmap_resized)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(pil_img)
    axes[0].set_title("Stimulus")
    axes[0].axis("off")

    axes[1].imshow(fmap_norm, cmap="inferno")
    axes[1].set_title(f"{layer_name}\nmean activation across all channels")
    axes[1].axis("off")

    axes[2].imshow(pil_img)
    axes[2].imshow(fmap_norm, cmap="inferno", alpha=alpha)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    title = (
        f"{condition} | {contour_type} | {layer_name} | "
        f"mean over {final_count[condition][contour_type]} J000 images"
    )
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()

    save_dir = MEAN_MAP_DIR / condition / contour_type
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"mean_overlay_{condition}_{contour_type}_{layer_name}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()
    plt.close(fig)


for condition in MODEL_CONDITIONS:
    for contour_type in ["C", "straight"]:
        for layer_name in BOTTOM_UP_LAYERS:
            plot_mean_state_overlay(condition, contour_type, layer_name, use_abs=USE_ABS_ACTIVATION)


In [ ]:
# ============================================================
# 10. Temporal dynamics: mean abs activation per layer/timestep
# ============================================================

records = []
for condition in MODEL_CONDITIONS:
    for contour in ["C", "straight"]:
        for layer in BOTTOM_UP_LAYERS:
            if temporal_sum[condition][contour][layer] is None:
                continue
            mean_stack = temporal_sum[condition][contour][layer] / temporal_count[condition][contour]
            for t in range(mean_stack.shape[0]):
                records.append({
                    "condition": condition,
                    "contour_type": contour,
                    "layer": layer,
                    "timestep": t,
                    "mean_abs_activation": mean_stack[t].abs().mean().item(),
                    "n_images": temporal_count[condition][contour],
                })

temporal_df = pd.DataFrame(records)
temporal_df.to_csv(OUTPUT_DIR / "csv" / "temporal_dynamics_full_vs_no_td.csv", index=False)
display(temporal_df.head())

plot_dir = OUTPUT_DIR / "plots" / "temporal_dynamics"
plot_dir.mkdir(parents=True, exist_ok=True)

for contour in ["C", "straight"]:
    for layer in BOTTOM_UP_LAYERS:
        sub = temporal_df[(temporal_df["contour_type"] == contour) & (temporal_df["layer"] == layer)]
        fig, ax = plt.subplots(figsize=(6, 4))
        for condition in MODEL_CONDITIONS:
            csub = sub[sub["condition"] == condition]
            ax.plot(csub["timestep"], csub["mean_abs_activation"], marker="o", label=condition)
        ax.set_title(f"Temporal dynamics | {contour} | {layer}")
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Mean abs activation")
        ax.legend()
        plt.tight_layout()
        save_path = plot_dir / f"temporal_{contour}_{layer}.png"
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)
        plt.show()
        plt.close(fig)


In [ ]:
# ============================================================
# 11. Mask metrics for all 128 h1_exc channels
# ============================================================


def quantify_all_channels_for_image_condition(model, row, condition, layer_name=ANALYSIS_LAYER):
    pil_img, img_tensor = load_image_tensor(row["path"])
    run_condition_forward(model, img_tensor, condition)
    state = get_state(model, layer_name).detach().cpu()

    contour_mask = load_true_contour_mask(row, target_size=pil_img.size)
    background_mask = ~contour_mask

    rows = []
    n_channels = state.shape[1]

    for channel in range(n_channels):
        fmap = single_channel_map(state, channel)
        fmap_resized = resize_map_to_image(fmap, pil_img)

        act = np.abs(fmap_resized) if USE_ABS_ACTIVATION else fmap_resized.copy()
        act = np.nan_to_num(act, nan=0.0, posinf=0.0, neginf=0.0)
        act = act - act.min() + 1e-8

        contour_values = act[contour_mask]
        background_values = act[background_mask]

        contour_mean = contour_values.mean()
        background_mean = background_values.mean()
        contour_sum = contour_values.sum()
        total_sum = act.sum()
        contour_area_pct = 100 * contour_mask.mean()

        contour_preference_ratio = contour_mean / (background_mean + 1e-8)
        activation_on_contour_pct = 100 * contour_sum / (total_sum + 1e-8)
        contour_enrichment = activation_on_contour_pct / (contour_area_pct + 1e-8)

        act_threshold = np.percentile(act, TOP_ACTIVATION_PERCENTILE)
        top_activation_mask = act >= act_threshold
        intersection = np.logical_and(contour_mask, top_activation_mask).sum()
        dice_top_activation = 2 * intersection / (contour_mask.sum() + top_activation_mask.sum() + 1e-8)

        rows.append({
            "condition": condition,
            "filename": row["filename"],
            "contour_type": row["contour_type"],
            "contrast": row["contrast"],
            "jitter": row["jitter"],
            "quadrant": row["quadrant"],
            "position": row["position"],
            "layer": layer_name,
            "channel": int(channel),
            "contour_mean_activation": contour_mean,
            "background_mean_activation": background_mean,
            "contour_preference_ratio": contour_preference_ratio,
            "activation_on_contour_pct": activation_on_contour_pct,
            "contour_area_pct": contour_area_pct,
            "contour_enrichment": contour_enrichment,
            "dice_top_activation": dice_top_activation,
            "top_activation_percentile": TOP_ACTIVATION_PERCENTILE,
        })

    return rows


metric_rows = []
for condition in MODEL_CONDITIONS:
    print("\n========================================")
    print("Condition:", condition)
    for _, row in tqdm(selected_df.iterrows(), total=len(selected_df)):
        metric_rows.extend(
            quantify_all_channels_for_image_condition(
                model=model,
                row=row,
                condition=condition,
                layer_name=ANALYSIS_LAYER,
            )
        )

mask_metrics_df = pd.DataFrame(metric_rows)
mask_metrics_df.to_csv(OUTPUT_DIR / "csv" / "mask_metrics_per_image_channel_full_vs_no_td.csv", index=False)
display(mask_metrics_df.head())


In [ ]:
# ============================================================
# 12. Mean per condition x contour x channel + summary
# ============================================================

mean_channel_df = (
    mask_metrics_df
    .groupby(["condition", "contour_type", "layer", "channel"], as_index=False)
    .agg(
        n_images=("filename", "nunique"),
        mean_contour_mean_activation=("contour_mean_activation", "mean"),
        sem_contour_mean_activation=("contour_mean_activation", "sem"),
        mean_background_mean_activation=("background_mean_activation", "mean"),
        sem_background_mean_activation=("background_mean_activation", "sem"),
        mean_contour_preference_ratio=("contour_preference_ratio", "mean"),
        sem_contour_preference_ratio=("contour_preference_ratio", "sem"),
        mean_activation_on_contour_pct=("activation_on_contour_pct", "mean"),
        sem_activation_on_contour_pct=("activation_on_contour_pct", "sem"),
        mean_contour_area_pct=("contour_area_pct", "mean"),
        mean_contour_enrichment=("contour_enrichment", "mean"),
        sem_contour_enrichment=("contour_enrichment", "sem"),
        mean_dice_top_activation=("dice_top_activation", "mean"),
        sem_dice_top_activation=("dice_top_activation", "sem"),
    )
)

mean_channel_df["rank_mean_contour_enrichment"] = (
    mean_channel_df
    .groupby(["condition", "contour_type", "layer"])["mean_contour_enrichment"]
    .rank(ascending=False, method="first")
    .astype(int)
)

mean_channel_df.to_csv(OUTPUT_DIR / "csv" / "mask_metrics_mean_per_channel_full_vs_no_td.csv", index=False)
display(mean_channel_df.head())

summary_df = (
    mean_channel_df
    .groupby(["condition", "contour_type", "layer"], as_index=False)
    .agg(
        n_channels=("channel", "count"),
        n_images=("n_images", "mean"),
        mean_contour_preference_ratio=("mean_contour_preference_ratio", "mean"),
        sem_contour_preference_ratio=("mean_contour_preference_ratio", "sem"),
        mean_activation_on_contour_pct=("mean_activation_on_contour_pct", "mean"),
        sem_activation_on_contour_pct=("mean_activation_on_contour_pct", "sem"),
        mean_contour_enrichment=("mean_contour_enrichment", "mean"),
        sem_contour_enrichment=("mean_contour_enrichment", "sem"),
        mean_dice_top_activation=("mean_dice_top_activation", "mean"),
        sem_dice_top_activation=("mean_dice_top_activation", "sem"),
    )
)

summary_df.to_csv(OUTPUT_DIR / "csv" / "mask_metrics_summary_full_vs_no_td.csv", index=False)
display(summary_df)


In [ ]:
# ============================================================
# 13. Compare full_td vs no_td: per-channel TD effect
# ============================================================

wide = mean_channel_df.pivot_table(
    index=["contour_type", "layer", "channel"],
    columns="condition",
    values=[
        "mean_contour_preference_ratio",
        "mean_activation_on_contour_pct",
        "mean_contour_enrichment",
        "mean_dice_top_activation",
    ],
).reset_index()

# flatten MultiIndex columns
wide.columns = [
    "_".join([str(x) for x in col if x != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in wide.columns
]

# Positive value means: full model > no-topdown model
wide["td_effect_contour_preference_ratio"] = (
    wide["mean_contour_preference_ratio_full_td"] -
    wide["mean_contour_preference_ratio_no_td"]
)
wide["td_effect_activation_on_contour_pct"] = (
    wide["mean_activation_on_contour_pct_full_td"] -
    wide["mean_activation_on_contour_pct_no_td"]
)
wide["td_effect_contour_enrichment"] = (
    wide["mean_contour_enrichment_full_td"] -
    wide["mean_contour_enrichment_no_td"]
)
wide["td_effect_dice_top_activation"] = (
    wide["mean_dice_top_activation_full_td"] -
    wide["mean_dice_top_activation_no_td"]
)

wide.to_csv(OUTPUT_DIR / "csv" / "td_effect_per_channel_full_minus_no_td.csv", index=False)
display(wide.head())

td_effect_summary = (
    wide
    .groupby(["contour_type", "layer"], as_index=False)
    .agg(
        n_channels=("channel", "count"),
        mean_td_effect_preference=("td_effect_contour_preference_ratio", "mean"),
        sem_td_effect_preference=("td_effect_contour_preference_ratio", "sem"),
        mean_td_effect_activation_pct=("td_effect_activation_on_contour_pct", "mean"),
        sem_td_effect_activation_pct=("td_effect_activation_on_contour_pct", "sem"),
        mean_td_effect_enrichment=("td_effect_contour_enrichment", "mean"),
        sem_td_effect_enrichment=("td_effect_contour_enrichment", "sem"),
        mean_td_effect_dice=("td_effect_dice_top_activation", "mean"),
        sem_td_effect_dice=("td_effect_dice_top_activation", "sem"),
    )
)

td_effect_summary.to_csv(OUTPUT_DIR / "csv" / "td_effect_summary_full_minus_no_td.csv", index=False)
display(td_effect_summary)


In [ ]:
# ============================================================
# 14. Figures: summary bars and per-channel TD effect boxplots
# ============================================================

PLOT_DIR = OUTPUT_DIR / "plots" / "td_effect"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# A. Full vs no-TD summary per contour
for metric in [
    "mean_contour_preference_ratio",
    "mean_activation_on_contour_pct",
    "mean_contour_enrichment",
    "mean_dice_top_activation",
]:
    fig, ax = plt.subplots(figsize=(8, 5))

    labels = []
    values = []
    sems = []

    for contour in ["C", "straight"]:
        for condition in ["full_td", "no_td"]:
            row = summary_df[
                (summary_df["contour_type"] == contour) &
                (summary_df["condition"] == condition)
            ].iloc[0]
            labels.append(f"{contour}\n{condition}")
            values.append(row[metric])
            sem_col = metric.replace("mean_", "sem_")
            sems.append(row[sem_col] if sem_col in row.index else 0)

    ax.bar(labels, values, yerr=sems)
    ax.set_ylabel(metric)
    ax.set_title(f"Full TD vs no TD | {metric} | {ANALYSIS_LAYER}")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()

    save_path = PLOT_DIR / f"summary_full_vs_no_td_{metric}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()
    plt.close(fig)

# B. Per-channel TD effect boxplot
metric = "td_effect_contour_enrichment"
fig, ax = plt.subplots(figsize=(7, 5))
labels = []
data = []
for contour in ["C", "straight"]:
    vals = wide[wide["contour_type"] == contour][metric].values
    labels.append(contour)
    data.append(vals)

ax.boxplot(data, labels=labels, showmeans=True)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_ylabel("TD effect on contour enrichment\nfull_td - no_td")
ax.set_title(f"Per-channel TD effect | {ANALYSIS_LAYER}")
plt.tight_layout()

save_path = PLOT_DIR / "boxplot_td_effect_contour_enrichment.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved:", save_path)
plt.show()
plt.close(fig)

# C. Scatter full vs no_td per channel
for contour in ["C", "straight"]:
    sub = wide[wide["contour_type"] == contour]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(
        sub["mean_contour_enrichment_no_td"],
        sub["mean_contour_enrichment_full_td"],
        alpha=0.7,
    )
    lim_min = min(sub["mean_contour_enrichment_no_td"].min(), sub["mean_contour_enrichment_full_td"].min())
    lim_max = max(sub["mean_contour_enrichment_no_td"].max(), sub["mean_contour_enrichment_full_td"].max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--")
    ax.set_xlabel("No TD contour enrichment")
    ax.set_ylabel("Full TD contour enrichment")
    ax.set_title(f"{contour} | per-channel enrichment")
    plt.tight_layout()
    save_path = PLOT_DIR / f"scatter_full_vs_no_td_enrichment_{contour}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# 15. Optional: classify C vs straight channels in full_td and test TD effect per channel group
# ============================================================

# Classificatie op basis van full_td h1_exc:
# C_channel als C enrichment - straight enrichment > 0.

full_h1 = mean_channel_df[
    (mean_channel_df["condition"] == "full_td") &
    (mean_channel_df["layer"] == ANALYSIS_LAYER)
].copy()

pref = full_h1.pivot(
    index="channel",
    columns="contour_type",
    values="mean_contour_enrichment",
).reset_index()

pref["C_minus_straight_enrichment"] = pref["C"] - pref["straight"]
pref["channel_group"] = np.where(
    pref["C_minus_straight_enrichment"] > 0,
    "C_channels",
    "straight_channels",
)

pref.to_csv(OUTPUT_DIR / "csv" / "channel_classification_full_td_h1.csv", index=False)

group_map = dict(zip(pref["channel"].astype(int), pref["channel_group"]))
wide["channel_group"] = wide["channel"].astype(int).map(group_map)

channel_group_effect = (
    wide
    .groupby(["contour_type", "layer", "channel_group"], as_index=False)
    .agg(
        n_channels=("channel", "count"),
        mean_td_effect_enrichment=("td_effect_contour_enrichment", "mean"),
        sem_td_effect_enrichment=("td_effect_contour_enrichment", "sem"),
        mean_td_effect_preference=("td_effect_contour_preference_ratio", "mean"),
        sem_td_effect_preference=("td_effect_contour_preference_ratio", "sem"),
        mean_td_effect_dice=("td_effect_dice_top_activation", "mean"),
        sem_td_effect_dice=("td_effect_dice_top_activation", "sem"),
    )
)

channel_group_effect.to_csv(OUTPUT_DIR / "csv" / "td_effect_by_channel_group.csv", index=False)
display(channel_group_effect)

# Plot channel group TD effect
for contour in ["C", "straight"]:
    sub = channel_group_effect[channel_group_effect["contour_type"] == contour]
    fig, ax = plt.subplots(figsize=(6, 4))
    labels = sub["channel_group"].tolist()
    values = sub["mean_td_effect_enrichment"].tolist()
    sems = sub["sem_td_effect_enrichment"].tolist()
    ax.bar(labels, values, yerr=sems)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_ylabel("TD effect on contour enrichment\nfull_td - no_td")
    ax.set_title(f"TD effect by channel group | {contour} | {ANALYSIS_LAYER}")
    plt.tight_layout()
    save_path = PLOT_DIR / f"td_effect_by_channel_group_{contour}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# 16. Quick interpretation helper
# ============================================================

print("Interpretatie:")
print("- td_effect = full_td - no_td")
print("- Positief betekent: de normale top-down pass verhoogt de metric t.o.v. bottom-up-only.")
print("- Negatief betekent: zonder top-down is de metric hoger, of top-down verlaagt die metric.")
print("- Voor jouw hypothese verwacht je vooral positieve td_effect_contour_enrichment in h1_exc.")

cols = [
    "contour_type",
    "mean_td_effect_enrichment",
    "sem_td_effect_enrichment",
    "mean_td_effect_preference",
    "sem_td_effect_preference",
    "mean_td_effect_dice",
    "sem_td_effect_dice",
]

display(td_effect_summary[cols])
